# Tracking Political Change with Embeddings of Parliamentary Speeches
### 2.2 Fine-tuning (speaker-contrastive, LoRA)
Fine-tune the Jina v3 backbone with LoRA using a speaker-identity contrastive loss, then check whether party separability improves on a held-out probe sample.

##### Import & Setup

In [1]:
%pip install --index-url https://download.pytorch.org/whl/cu124 --extra-index-url https://pypi.org/simple \
    torch==2.6.0 torchvision torchaudio transformers==5.9.0 accelerate peft scikit-learn

Looking in indexes: https://download.pytorch.org/whl/cu124, https://pypi.org/simple
  Using cached https://download-r2.pytorch.org/whl/cu124/torch-2.6.0%2Bcu124-cp311-cp311-linux_x86_64.whl.metadata (28 kB)
  Using cached torchvision-0.28.0-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (5.6 kB)
  Using cached torchaudio-2.11.0-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (6.9 kB)
  Using cached transformers-5.9.0-py3-none-any.whl.metadata (33 kB)
  Using cached accelerate-1.14.0-py3-none-any.whl.metadata (19 kB)
  Using cached peft-0.20.0-py3-none-any.whl.metadata (14 kB)
  Using cached filelock-3.32.3-py3-none-any.whl.metadata (2.0 kB)
  Using cached typing_extensions-4.16.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.4.127-py3-none-manylinux2014_x86_64.

In [1]:
import time
import random
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import LabelEncoder
from peft import LoraConfig, get_peft_model
from embedding_utils import mean_pooling, embed_speeches

In [2]:
SEED = 24
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


##### Load model & attach LoRA

In [3]:
MODEL_NAME = "jinaai/jina-embeddings-v3-hf"                 # base model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModel.from_pretrained(MODEL_NAME).to(device)
hidden_size = base_model.config.hidden_size

Loading weights:   0%|          | 0/294 [00:00<?, ?it/s]

In [4]:
# LoRA configuration on the attention projections 
lora_config = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"])

# sanity check: verifies target_modules matched the backbone's actual layer names
encoder = get_peft_model(base_model, lora_config)
n_trainable = sum(p.numel() for p in encoder.parameters() if p.requires_grad)
assert n_trainable > 0, "LoRA attached to zero modules — check target_modules against named_modules()"
encoder.print_trainable_parameters()

trainable params: 1,572,864 || all params: 560,936,960 || trainable%: 0.2804


In [5]:
# Load the pre-built fine-tuning and party-probe splits 
df_subsample = pd.read_csv("finetune_subsample.csv")
probe_val_df = pd.read_csv("probe_val_sample.csv")
probe_test_df = pd.read_csv("probe_test_sample.csv")

# speech and speaker counts for the fine-tuning subsample
print(f"Fine-tuning subsample: {len(df_subsample)} speeches | "
      f"{df_subsample['politicianId'].nunique()} speakers")
print("\n")

# outputs speech counts per party for fine-tuning and probe samples
print("Party breakdown in fine-tuning subsample:")
print(df_subsample["party"].value_counts())
print("-"*30)
print(f"Party-probe sample: {len(probe_val_df)} speeches")
print(probe_val_df["party"].value_counts())

finetune_speech_ids = set(df_subsample["id"])
print(f"Contrastive train: {len(df_subsample)}")

Fine-tuning subsample: 22513 speeches | 1216 speakers


Party breakdown in fine-tuning subsample:
party
CDU/CSU         7011
SPD             5265
Grüne           3477
FDP             2901
LINKE           2649
AfD              858
PDS              264
Fraktionslos      88
Name: count, dtype: int64
------------------------------
Party-probe sample: 720 speeches
party
AfD             90
CDU/CSU         90
FDP             90
Fraktionslos    90
Grüne           90
LINKE           90
PDS             90
SPD             90
Name: count, dtype: int64
Contrastive train: 22513


##### Speaker-grouped (PK) batching

In [6]:
# turns politicianId into 0 to N-1 integer
def encode_speaker_labels(df):
    speaker_ids = sorted(df["politicianId"].unique())
    speaker_to_label = {sid: i for i, sid in enumerate(speaker_ids)}
    return df["politicianId"].map(speaker_to_label).to_numpy()      

# reset the index so that row position lines up with the position of train_label
df_subsample = df_subsample.reset_index(drop=True)
train_labels = encode_speaker_labels(df_subsample)

# builds batches from P speakers with up to K speeches each, instead of random shuffling (which wouldn't give positive pairs)
class PKBatchSampler:
    """Yields P*K-index batches: P speakers, up to K speeches each. Speakers with
    fewer than K speeches just contribute what they've got (always >= 2, since
    we only sample from speakers with at least 2 speeches in this split.)"""
    def __init__(self, labels, P=8, K=4, seed=SEED):
        self.P, self.K = P, K
        self.rng = random.Random(seed)
        by_speaker = defaultdict(list)
        for idx, lab in enumerate(labels):
            by_speaker[lab].append(idx)     # group speech indices by speaker
        self.by_speaker = {lab: idxs for lab, idxs in by_speaker.items() if len(idxs) >= 2} # only speakers with >= 2 speeches can supply a positive pair
        self.speakers = list(self.by_speaker.keys())
        n_batches_est = max(1, len(self.speakers) // self.P)    # rough estimate (some chunks get skipped in __iter__, so actual batch count can be lower)
        self.n_batches = n_batches_est

    def __iter__(self):
        pools = {lab: list(idxs) for lab, idxs in self.by_speaker.items()}
        for pool in pools.values():
            self.rng.shuffle(pool)
        speaker_order = list(self.speakers)
        self.rng.shuffle(speaker_order)

        for i in range(0, len(speaker_order), self.P):
            chunk = speaker_order[i:i + self.P]
            if len(chunk) < 2:
                continue
            batch = []
            for lab in chunk:
                take = pools[lab][:self.K]
                if len(take) >= 2:
                    batch.extend((idx, lab) for idx in take)
            if len(batch) >= 4:  # need at least 2 speakers x 2 to have both pos and neg
                yield batch

    def __len__(self):
        return self.n_batches

##### Dataset & collate

In [9]:
# checks how much truncation happens at a few candidate max_lengths
lengths = df_subsample["speechContent"].apply(lambda t: len(tokenizer.encode(t)))
print(lengths.describe())

for max_len in [128, 256, 512, 1024]:
    print(f"% truncated at {max_len}: {(lengths > max_len).mean():.1%}")

BEST_LENGTH = 1024       # heavy truncation even here (40.3%) - GPU/time budget constraint, not full coverage

count    22513.000000
mean       915.196864
std        614.896892
min        100.000000
25%        395.000000
50%        907.000000
75%       1230.000000
max       8194.000000
Name: speechContent, dtype: float64
% truncated at 128: 94.6%
% truncated at 256: 81.0%
% truncated at 512: 71.5%
% truncated at 1024: 40.3%


In [12]:
class SpeechDataset(Dataset):
    """Wraps speech texts, indexed by (idx, label) pairs from PKBatchSampler."""
    def __init__(self, df):
        self.texts = df["speechContent"].tolist()

    def __getitem__(self, idx_and_label):
        idx, label = idx_and_label
        return self.texts[idx], label

    def __len__(self):
        return len(self.texts)

# tokenizes one batch of texts
def make_collate_fn(tokenizer, max_length=BEST_LENGTH):      # shorter than the 8192 used for full-speech embedding (speeches get truncated for training)
    def collate(batch):
        texts, labels = zip(*batch)
        enc = tokenizer(
            list(texts), padding=True, truncation=True,
            max_length=max_length, return_tensors="pt")
        return enc, torch.tensor(labels, dtype=torch.long)
    return collate

train_dataset = SpeechDataset(df_subsample)

P, K = 8, 4  # 8 speakers with up to 4 speeches per batch
train_sampler = PKBatchSampler(train_labels, P=P, K=K, seed=SEED)
collate_fn = make_collate_fn(tokenizer, max_length=BEST_LENGTH)

# combines sampler, dataset, and collate into one iterable
def make_loader(dataset, sampler, collate_fn):
    for batch in sampler:
        yield collate_fn([dataset[idx_lab] for idx_lab in batch])

##### Batch embedding & loss

In [13]:
# runs the model, pools, normalizes
def embed_batch(model, enc, device):
    enc = {k: v.to(device) for k, v in enc.items()}
    out = model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"])
    pooled = mean_pooling(out.last_hidden_state, enc["attention_mask"])
    return F.normalize(pooled, p=2, dim=1)

# infoNCE loss: same speaker in the batch = positive, everyone else = negative
def supervised_contrastive_loss(embeddings, labels, temperature=0.05):
    device = embeddings.device
    sim = embeddings @ embeddings.T / temperature          # (B, B)
    labels = labels.to(device)
    same_label = labels.unsqueeze(0) == labels.unsqueeze(1)  # (B, B)
    self_mask = torch.eye(len(labels), dtype=torch.bool, device=device)
    positive_mask = same_label & ~self_mask

    has_positive = positive_mask.any(dim=1)      # rows with no positive in this batch can't contribute a defined loss
    if has_positive.sum() == 0:
        return torch.tensor(0.0, device=device, requires_grad=True)

    sim = sim.masked_fill(self_mask, float("-inf"))             # exclude self-similarity from softmax
    log_prob = sim - torch.logsumexp(sim, dim=1, keepdim=True)  # log-softmax over each row

    pos_log_prob = log_prob.masked_fill(~positive_mask, 0.0).sum(dim=1)
    n_pos = positive_mask.sum(dim=1).clamp(min=1)       # avoid divide-by-zero for has_positive check above
    per_anchor_loss = -(pos_log_prob / n_pos)           # average log-prob over each anchor's positives

    return per_anchor_loss[has_positive].mean()

##### Party-probe evaluation

In [14]:
# embeds the held-out probe set and checks if party separates (classifier acc + silhouette)
def evaluate_party_probe(model, tokenizer, probe_val_df, device, batch_size=16, max_length=512):
    model.eval()            # disable dropout etc. for a stable eval pass
    texts = probe_val_df["speechContent"].tolist()
    labels = LabelEncoder().fit_transform(probe_val_df["party"])

    all_emb = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            enc = tokenizer(batch, padding=True, truncation=True,
                             max_length=max_length, return_tensors="pt").to(device)
            out = model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"])
            pooled = mean_pooling(out.last_hidden_state, enc["attention_mask"])
            emb = F.normalize(pooled, p=2, dim=1)
            all_emb.append(emb.cpu().numpy())
    emb = np.vstack(all_emb)

    probe_acc = cross_val_score(
        LogisticRegression(max_iter=1000), emb, labels,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)).mean()
    
    sil = silhouette_score(emb, labels)
    model.train()       # restore training mode for the next training step
    return probe_acc, sil

##### Training loop

In [15]:
N_EPOCHS = 3
LR = 2e-5
EVAL_EVERY = 50
TEMPERATURE = 0.05

# only LoRA params are trainable, base weights stay frozen
optimizer = torch.optim.AdamW([p for p in encoder.parameters() if p.requires_grad], lr=LR)

# party-probe accuracy/silhouette before fine-tuning
baseline_acc, baseline_sil = evaluate_party_probe(encoder, tokenizer, probe_val_df, device)
print(f"[step 0 / off-the-shelf] party-probe acc={baseline_acc:.4f} | silhouette={baseline_sil:.4f}")

best_score = baseline_acc  # only keep a checkpoint if it beats off-the-shelf
best_state = None
history = []
global_step = 0     # counts across all epochs, not reset per epoch
t0 = time.time()

for epoch in range(N_EPOCHS):
    for enc, labels in make_loader(train_dataset, train_sampler, collate_fn):
        encoder.train()
        embeddings = embed_batch(encoder, enc, device)
        loss = supervised_contrastive_loss(embeddings, labels, temperature=TEMPERATURE)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        global_step += 1
        history.append({"step": global_step, "epoch": epoch, "train_loss": loss.item()})

        if global_step % EVAL_EVERY == 0:
            acc, sil = evaluate_party_probe(encoder, tokenizer, probe_val_df, device)
            history[-1].update({"probe_acc": acc, "probe_sil": sil})
            print(f"epoch {epoch+1} step {global_step} | contrastive_loss={loss.item():.4f} "
                  f"| party-probe acc={acc:.4f} silhouette={sil:.4f} | best so far={best_score:.4f}")

            if acc > best_score:
                best_score = acc
                best_state = {k: v.detach().clone() for k, v in encoder.state_dict().items()
                              if "lora_" in k}      # only the small LoRA weights, not the full backbone
                print(f"  -> new best (acc={best_score:.4f}), checkpointing LoRA state")

print(f"Training done in {(time.time()-t0)/60:.1f} min | best probe acc = {best_score:.4f} "
      f"(off-the-shelf was {baseline_acc:.4f})")

[step 0 / off-the-shelf] party-probe acc=0.3153 | silhouette=-0.0081
epoch 1 step 50 | contrastive_loss=3.4964 | party-probe acc=0.3264 silhouette=-0.0079 | best so far=0.3153
  -> new best (acc=0.3264), checkpointing LoRA state
epoch 1 step 100 | contrastive_loss=2.9160 | party-probe acc=0.3444 silhouette=-0.0078 | best so far=0.3264
  -> new best (acc=0.3444), checkpointing LoRA state
epoch 1 step 150 | contrastive_loss=2.9060 | party-probe acc=0.3458 silhouette=-0.0082 | best so far=0.3444
  -> new best (acc=0.3458), checkpointing LoRA state
epoch 2 step 200 | contrastive_loss=2.7885 | party-probe acc=0.3736 silhouette=-0.0084 | best so far=0.3458
  -> new best (acc=0.3736), checkpointing LoRA state
epoch 2 step 250 | contrastive_loss=2.9194 | party-probe acc=0.3972 silhouette=-0.0087 | best so far=0.3736
  -> new best (acc=0.3972), checkpointing LoRA state
epoch 2 step 300 | contrastive_loss=2.5462 | party-probe acc=0.4069 silhouette=-0.0094 | best so far=0.3972
  -> new best (acc=

##### Finalize best checkpoint & compare with off-the-shelf

In [16]:
history_df = pd.DataFrame(history)
history_df.to_csv("contrastive_finetune_history.csv", index=False)

if best_state is not None:
    encoder.load_state_dict(best_state, strict=False)       # best_state only has LoRA keys, not the full state dict
    print("Loaded best-probe-score LoRA checkpoint into the model.")
else:
    print("WARNING: no checkpoint beat off-the-shelf, keeping backbone unmodified.")

# Post-training, pre-merge accuracy/silhouette (to later compare against post-merge)
final_acc, final_sil = evaluate_party_probe(encoder, tokenizer, probe_val_df, device)
print(f"[pre-merge] party-probe acc={final_acc:.4f} | silhouette={final_sil:.4f}")

Loaded best-probe-score LoRA checkpoint into the model.
[pre-merge] party-probe acc=0.4264 | silhouette=-0.0079


In [17]:
FT_BACKBONE_PATH = "jina_v3_contrastive_backbone"

if best_state is not None:
    merged_model = encoder.merge_and_unload()       # folds LoRA weights into the base model, drops the adapter wrapper
    used_finetuned_weights = True
else:
    merged_model = base_model  # unmodified off-the-shelf backbone
    used_finetuned_weights = False

merged_model.save_pretrained(FT_BACKBONE_PATH)
tokenizer.save_pretrained(FT_BACKBONE_PATH)
print(f"Saved backbone to {FT_BACKBONE_PATH} | used_finetuned_weights={used_finetuned_weights}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved backbone to jina_v3_contrastive_backbone | used_finetuned_weights=True


In [18]:
# loads model, embeds texts, then frees GPU memory (used to compare two models without holding both in memory at once)
def embed_with_model(model_path, texts, batch_size=16, max_length=512, device=device):
    tok = AutoTokenizer.from_pretrained(model_path)
    mdl = AutoModel.from_pretrained(model_path).to(device)
    mdl.eval()

    all_emb = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        inputs = tok(batch, padding=True, truncation=True, max_length=max_length,
                     return_tensors="pt").to(device)
        with torch.no_grad():
            out = mdl(**inputs)
            emb = mean_pooling(out.last_hidden_state, inputs["attention_mask"])
            emb = F.normalize(emb, p=2, dim=1)
        all_emb.append(emb.cpu().numpy())

    del mdl     # frees this model before the caller loads the next one
    torch.cuda.empty_cache()
    return np.vstack(all_emb)

# classifier accuracy + silhouette score, the two probe metrics used throughout
def evaluate_embeddings(emb, labels, name):
    probe_acc = cross_val_score(
        LogisticRegression(max_iter=1000), emb, labels,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)).mean()
    
    sil = silhouette_score(emb, labels)
    print(f"{name}: probe accuracy = {probe_acc:.4f} | silhouette = {sil:.4f}")
    return probe_acc, sil

In [19]:
eval_texts = probe_test_df["speechContent"].tolist()
eval_labels = LabelEncoder().fit_transform(probe_test_df["party"])

# re-embeds the same probe set with both models, so the comparison below isn't affected by the earlier pre-merge numbers
offtheshelf_emb = embed_with_model(MODEL_NAME, eval_texts)
contrastive_emb = embed_with_model(FT_BACKBONE_PATH, eval_texts)

offtheshelf_acc, offtheshelf_sil = evaluate_embeddings(offtheshelf_emb, eval_labels, "Off-the-shelf")
contrastive_acc, contrastive_sil = evaluate_embeddings(contrastive_emb, eval_labels, "Contrastive fine-tuned")

# Sanity check: merge + save + reload from disk shouldn't have changed accuracy
print(f"merge sanity check: pre-merge acc={final_acc:.4f} vs post-merge acc={contrastive_acc:.4f} "
      f"(diff={contrastive_acc - final_acc:+.4f}, should be small (CV noise only).)")

Loading weights:   0%|          | 0/294 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/294 [00:00<?, ?it/s]

Off-the-shelf: probe accuracy = 0.3062 | silhouette = -0.0074
Contrastive fine-tuned: probe accuracy = 0.4021 | silhouette = -0.0089
merge sanity check: pre-merge acc=0.4264 vs post-merge acc=0.4021 (diff=-0.0243, should be small (CV noise only).)


##### Embed the fine tuned model

In [20]:
# reload the saved backbone (fine-tuned if a checkpoint beat baseline, off-the-shelf otherwise) rather than reusing merged_model in memory
ft_model = AutoModel.from_pretrained(FT_BACKBONE_PATH).to(device)
ft_model.eval()
embedding_df = pd.read_csv("embedding_corpus.csv")  # full corpus to re-embed, disjoint from the fine-tuning subsample

Loading weights:   0%|          | 0/294 [00:00<?, ?it/s]

In [21]:
# same embedding call as the baseline, just with ft_model instead of the off-the-shelf model
texts = embedding_df["speechContent"].tolist()
print(f"Embedding {len(texts)} speeches with the contrastive-fine-tuned encoder...")

t0 = time.time()

embeddings_ft = embed_speeches(texts, ft_model, tokenizer, device, batch_size=32,
                                checkpoint_path="finetuned_checkpoint.npy")

elapsed = time.time() - t0

rate = len(texts) / elapsed
print(f"Shape: {embeddings_ft.shape}")
print(f"Time: {elapsed/60:.1f} min | Rate: {rate:.2f} speeches/sec")

Embedding 82447 speeches with the contrastive-fine-tuned encoder...
Embedded 32/82447
Embedded 64/82447
Embedded 96/82447
Embedded 128/82447
Embedded 160/82447
Embedded 192/82447
Embedded 224/82447
Embedded 256/82447
Embedded 288/82447
Embedded 320/82447
Embedded 352/82447
Embedded 384/82447
Embedded 416/82447
Embedded 448/82447
Embedded 480/82447
Embedded 512/82447
Embedded 544/82447
Embedded 576/82447
Embedded 608/82447
Embedded 640/82447
Embedded 672/82447
Embedded 704/82447
Embedded 736/82447
Embedded 768/82447
Embedded 800/82447
Embedded 832/82447
Embedded 864/82447
Embedded 896/82447
Embedded 928/82447
Embedded 960/82447
Embedded 992/82447
Embedded 1024/82447
Embedded 1056/82447
Embedded 1088/82447
Embedded 1120/82447
Embedded 1152/82447
Embedded 1184/82447
Embedded 1216/82447
Embedded 1248/82447
Embedded 1280/82447
Embedded 1312/82447
Embedded 1344/82447
Embedded 1376/82447
Embedded 1408/82447
Embedded 1440/82447
Embedded 1472/82447
Embedded 1504/82447
Embedded 1536/82447
Embedd

##### Save the fine tuned model

In [22]:
df_meta = embedding_df.copy()
df_meta["used_in_finetune"] = False  
df_meta["used_in_party_probe_val"] = df_meta["id"].isin(set(probe_val_df["id"]))
df_meta["used_in_party_probe_test"] = df_meta["id"].isin(set(probe_test_df["id"]))
df_meta["used_finetuned_weights"] = used_finetuned_weights
df_meta["embedding"] = list(embeddings_ft)

df_meta.to_parquet("jina_v3_contrastive_full.parquet", engine="pyarrow", index=False)

print("Saved embeddings:", embeddings_ft.shape)
print("Saved rows:", len(df_meta))
print(f"Rows used in party probe (val):  {df_meta['used_in_party_probe_val'].sum()}")
print(f"Rows used in party probe (test): {df_meta['used_in_party_probe_test'].sum()}")
print(f"used_finetuned_weights: {used_finetuned_weights}")

Saved embeddings: (82447, 1024)
Saved rows: 82447
Rows used in party probe (val):  720
Rows used in party probe (test): 480
used_finetuned_weights: True
